# File Search Store Management

A notebook to create and manage the Gemini FileSearchStore of prisoner correspondence.

## Pre-Reqs and Notes

- The `file_search_stores` is a feature exclusive to the Gemini Developer API. 
  - It does not work with the Vertex AI API or the Gen AI SDK in Vertex AI mode.
  - Therefore: don't set env vars for `GOOGLE_CLOUD_LOCATION` or `GOOGLE_GENAI_USE_VERTEXAI` and do not initialise Vertex AI.
- Make sure you have an up-to-date version of the `google-genai` package installed. 
  - Versions older than 1.49.0 do not support the File Search Tool.
  - If using `pip`: `pip install --upgrade google-genai`.
  - You can add to your `pyproject.toml` file; since we don't explicitly need it outside
- Add the Gemini API key as an environment variable. Then use `source .env` to export the environment variables to the current session. 

## Setup

In [ ]:
import glob
import os
import textwrap
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

Load environment variables.

In [ ]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GEMINI_API_KEY:
    print("Warning: GOOGLE_API_KEY environment variable not set. Client may fail.")
else:
    print("Successfully loaded Gemini API key.")

STORE_NAME = os.getenv("STORE_NAME")
if not STORE_NAME:
    print("Warning: STORE_NAME environment variable not set. You will not be able to work with the store.")
else:
    print(f"Successfully loaded Gemini File Search Store: {STORE_NAME}")

MODEL = os.getenv("MODEL")
if not MODEL:
    print("Warning: MODEL environment variable not set.")
else:
    print(f"Successfully loaded model: {MODEL}")

Client initialization.

In [ ]:
client = genai.Client()

## Store Management

View all stores.

In [ ]:
try:
    for store in client.file_search_stores.list():
        print(store)
except Exception as e:
    print(f"Error listing stores (check creds?): {e}")

Retrieve the store(s) that match a given display name. Note that display name is not unique, so this function returns the first matching store.

In [ ]:
def get_store(store_name: str) -> genai.types.FileSearchStore | None:
    """Retrieve a store by its display name."""
    try:
        for store in client.file_search_stores.list():
            if store.display_name == store_name:
                return store
    except Exception as e:
        print(f"Error in get_store path: {e}")
    return None

Create the prisoner correspondence store (one time).

In [ ]:
if STORE_NAME is None:
    print("STORE_NAME is not set. Cannot create store.")
elif not get_store(STORE_NAME):
    file_search_store = client.file_search_stores.create(config={"display_name": STORE_NAME})
    print(f"Created store: {file_search_store.name}")
else:
    print(f"Store {STORE_NAME} already exists.")

Interrogate a store and see what files have been uploaded to it.

In [ ]:
def view_store_files(store_name: str):
    file_search_store = get_store(store_name)
    if not file_search_store:
        print(f"Store {store_name} not found.")
    elif file_search_store.name is not None:
        print(f"Store found: {file_search_store.display_name}")
        docs = client.file_search_stores.documents.list(parent=file_search_store.name)
        try:
            doc_list = list(docs)
            print(f"Docs in {store_name}: {len(doc_list)}")
            if not doc_list:
                print("No documents found in the store.")
            else:
                for i, doc in enumerate(doc_list):
                    section_heading = f"Document {i}:"
                    print("-" * len(section_heading))
                    print(section_heading)
                    print("-" * len(section_heading))
                    print(f"  Display name:{doc.display_name}")
                    print(f"  ID: {doc.name}")
                    print(f"  Metadata: {doc.custom_metadata}")
        except Exception as e:
            print(f"Error listing docs (might be empty): {e}")
    else:
        print(f"Failed to find resource name for {file_search_store.display_name}.")

if STORE_NAME is not None:
    view_store_files(STORE_NAME)

Delete a store.

In [ ]:
def delete_store(store_name: str):
    store_to_delete = get_store(store_name)
    if store_to_delete and store_to_delete.name is not None:
        print(f"Deleting: {store_to_delete.name}")
        client.file_search_stores.delete(name=store_to_delete.name, config={'force': True})
    else:
        print("Store not found.")

## Upload and Process Files

Set path to the folder containing the files we want to upload.

In [ ]:
UPLOAD_PATH = "../data"

Create a utility class for document metadata.

In [ ]:
class DocumentMetadata(BaseModel):
    """Metadata for a document"""
    title: str = Field(max_length=255)
    author: str = Field(max_length=255)
    abstract: str = Field(max_length=255)

We will use Gemini to create metadata for each document.

In [ ]:
def generate_metadata(file_name: str, temp_file, model:str) -> DocumentMetadata:
    """Generate metadata for a document, using Gemini to extract title, author, and abstract."""
    print(f"Generating metadata for {file_name}...")
    response = client.models.generate_content(
        model=model,
        contents=[
            """Please extract title, author, and short abstract from this document. 
            Each value should be under 200 characters.

            Abstracts should be succinct and NOT include preamble text like `This document describes...`

            Example bad abstract: 
            This letter discusses the writer's concerns about obtaining a phone
            and finding an affordable carrier plan after his release from prison.

            Example good abstract:
            Questions about obtaining a phone and selecting an affordable prepaid 
            carrier plan upon release.

            Example bad abstract:
            This document describes the prisoner's interest in learning filmmaking
            and his hopes to attend film school.

            Example good abstract:
            Interest in learning iPhone filmmaking and applying to University of 
            Michigan film school to create social impact documentaries.
            """,
            temp_file,
        ],
        config={
            "response_mime_type": "application/json",
            "response_schema": DocumentMetadata,
        },
    )
    metadata = DocumentMetadata.model_validate(response.parsed)
    print(f"Title: {metadata.title}")
    print(f"Author: {metadata.author}")
    print(f"Abstract: {metadata.abstract}")
    return metadata

Utility functions to upload and delete documents from our store.

In [ ]:
def delete_doc(doc):
    """
    Delete a document from its file search store.
    Note that the doc already references its file search store.
    So we don't need to pass the file search store name.
    """
    print(f"Deleting document: '{doc.display_name}' (ID: {doc.name})")
    client.file_search_stores.documents.delete(name=doc.name, config={"force": True})
    time.sleep(2)  # small throttle to allow propagation


def upload_doc(file_path, file_search_store: genai.types.FileSearchStore, model: str, upload_timeout: int = 300):
    """Upload a document to the file search store
    
    Args:
        file_path: Path to the file to upload
        file_search_store: The file search store to upload to
        model: The model to use for metadata extraction
        upload_timeout: Maximum time in seconds to wait for upload (default: 300)
    """
    file_name = os.path.basename(file_path)

    if file_search_store.name is None:
        raise RuntimeError("File search store does not have a valid resource name.")

    # Check if this is a replacement of an existing file and skip if so 
    for doc in client.file_search_stores.documents.list(parent=file_search_store.name):
        if doc.display_name == file_name:
            return

    # Now upload the file for metadata extraction
    print(f"Uploading {file_name} for metadata extraction...")
    temp_file = client.files.upload(file=file_path)
    if temp_file.name is None:
        raise RuntimeError(f"File upload failed: no file name assigned for {file_name}")

    # Verify file is active (ready for inference) with timeout
    start_time = time.time()
    while temp_file.state == genai.types.FileState.PROCESSING:
        elapsed = time.time() - start_time
        if elapsed > upload_timeout:
            raise TimeoutError(
                f"File upload timed out after {upload_timeout} seconds for {file_name}"
            )
        time.sleep(2)
        temp_file = client.files.get(name=temp_file.name) # type: ignore
    if temp_file.state != genai.types.FileState.ACTIVE:
        raise RuntimeError(f"File upload failed with state: {temp_file.state}")
    
    print(f"\n{file_name} upload complete, processing metadata...")
    metadata = generate_metadata(file_name, temp_file, model)

    # Import the file into the file search store with custom metadata
    print(f"Indexing {file_name} in file search store...")
    operation = client.file_search_stores.upload_to_file_search_store(
        file_search_store_name=file_search_store.name,
        file=file_path,
        config={
            "display_name": file_name,
            "custom_metadata": [
                {"key": "title", "string_value": metadata.title},
                {"key": "file_name", "string_value": file_name},
                {"key": "author", "string_value": metadata.author},
                {"key": "abstract", "string_value": metadata.abstract},
            ],
        },
    )

    # Wait until import is complete with timeout
    index_start = time.time()
    while not operation.done:
        elapsed = time.time() - index_start
        if elapsed > upload_timeout:
            raise TimeoutError(
                f"File indexing timed out after {upload_timeout} seconds for {file_name}"
            )
        time.sleep(5)
        operation = client.operations.get(operation)

    print(f"{file_name} successfully uploaded and indexed")


Now actually **upload and process our documents**:

In [ ]:
if STORE_NAME is None or MODEL is None:
    print("STORE_NAME or MODEL is not set. Cannot upload files.")
else:
    file_search_store = get_store(STORE_NAME)
    if file_search_store is None:
        print(f"Store {STORE_NAME} not found.")
    else:
        print(f"Uploading files to {file_search_store.name}...")
        files_to_upload = glob.glob(f"{UPLOAD_PATH}/*")
        if files_to_upload:
            for file_path in files_to_upload:
                print(f"Uploading {os.path.basename(file_path)}")
                upload_doc(file_path, file_search_store, MODEL)
            print("Upload complete.")
        else:
            print(f"No files found in {UPLOAD_PATH}")

## Verify with Query

Now that the data is uploaded, let's verify we can retrieve it using the File Search Tool.

In [ ]:
if STORE_NAME is None or MODEL is None:
    print("STORE_NAME or MODEL is not set. Cannot upload files.")
else:
    store = get_store(STORE_NAME)
    question = "Please create a re-entry plan for the client. Summarize current situation and needs. Use the file search tool if you need to reference documents in the store."
    if store and store.name is not None:
        print(f"Querying store: {store.name} ({store.display_name})")
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=question,
                config=types.GenerateContentConfig(
                    tools=[types.Tool(file_search=types.FileSearch(file_search_store_names=[store.name]))]
                ),
            )
            print("\nResponse:")
            wrapped_text = textwrap.fill(response.text, width=100) # type: ignore
            print(wrapped_text)
        except Exception as e:
            print(f"Query failed: {e}")
    else:
        print("Store not found, cannot verify.")